<a href="https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)


HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

print("Setup complete.")

Setup complete.


In [14]:
df = con.sql(f"""
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        month,
        SUM(gsc_impressions) AS impressions
    FROM {TABLES['fact_daily']}
    WHERE gsc_data_available IS TRUE
      AND month IN ('2026-02', '2026-03')
    GROUP BY client_hash_id, content_hash_id, month
),

momentum AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE
            WHEN month = '2026-02' THEN impressions ELSE 0
        END) AS prev_month_impressions,

        SUM(CASE
            WHEN month = '2026-03' THEN impressions ELSE 0
        END) AS current_month_impressions

    FROM monthly
    GROUP BY client_hash_id, content_hash_id
),

query_signals AS (
    SELECT
        content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_queries,
        ANY_VALUE(rare_impressions_share) AS rare_share,
        ANY_VALUE(anonymized_impressions_share) AS anon_share,
        MAX(impressions_90d) AS top_query_impressions,
        SUM(impressions_90d) AS kept_impressions
    FROM read_parquet(
        '{REL}/fact_content_query_90d.parquet'
    )
    GROUP BY content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.prev_month_impressions,
    m.current_month_impressions,

    q.visible_queries,
    q.rare_share,
    q.anon_share,

    q.top_query_impressions /
        NULLIF(q.kept_impressions, 0) AS top_query_share,

    CASE
        WHEN m.current_month_impressions
             < 0.8 * m.prev_month_impressions
        THEN 1
        ELSE 0
    END AS target

FROM momentum m

LEFT JOIN query_signals q
    ON m.content_hash_id = q.content_hash_id

WHERE m.prev_month_impressions > 0
""").df()

print("Rows:", len(df))
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 153559


,client_hash_id,content_hash_id,prev_month_impressions,current_month_impressions,visible_queries,rare_share,anon_share,top_query_share,target
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,386.0,247.0,14,0.043656,0.725102,0.162242,1
1,client_08a6a72ff48e62c0,content_4486e5efcc7b773f,553.0,505.0,10,0.041108,0.748594,0.347737,0
2,client_08a6a72ff48e62c0,content_44c082bdb9a864ea,57.0,160.0,23,0.004720,0.810897,0.167181,0
3,client_08a6a72ff48e62c0,content_44c8757dd2f454d0,70.0,79.0,1,0.101695,0.389831,1.000000,0
4,client_08a6a72ff48e62c0,content_44d471e5c97d8fdc,7.0,2.0,2,0.469388,0.040816,0.500000,1


In [15]:
feature_cols = [
    "prev_month_impressions",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

model_data = df.dropna(subset=feature_cols).copy()

X = model_data[feature_cols]
y = model_data["target"]
groups = model_data["client_hash_id"]

print("Rows:", len(model_data))
print("Features:", feature_cols)
print("\nTarget distribution:")
print(y.value_counts(normalize=True))

Rows: 85475
Features: ['prev_month_impressions', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']

Target distribution:
target
0    0.857397
1    0.142603
Name: proportion, dtype: float64


In [16]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

Training rows: 54384
Test rows: 31091
Training clients: 29
Test clients: 8


In [17]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

prob = model.predict_proba(X_test)[:, 1]

print("Model trained.")

Model trained.


## 1. Ranked actions + reason codes

The queue is used to prioritise content for human review. It is based on the model score and supporting search-performance signals.

A higher model score means the content received a stronger directional decline signal from the model. It does not mean that a decline will definitely occur.

### Reason codes

| Reason code | Meaning | Suggested action |
|---|---|---|
| DECLINE_SIGNAL | Higher model score for a possible decline | Review content and recent performance |
| LOW_VISIBILITY | Low previous-month impressions | Monitor before prioritising a refresh |
| QUERY_CONCENTRATION | A large share of impressions comes from the top query | Review query coverage and search intent |
| HIGH_QUERY_COVERAGE | Content has a higher number of visible queries | Check whether the content still covers the queries |
| MONITOR | No strong signal from the available features | Continue monitoring |

The recommended action is a starting point for review. A human should make the final decision.

In [18]:
queue = model_data.iloc[test_idx].copy()

queue["model_score"] = prob

def get_reason_codes(row):
    reasons = []

    if row["model_score"] >= 0.70:
        reasons.append("DECLINE_SIGNAL")

    if row["prev_month_impressions"] < 100:
        reasons.append("LOW_VISIBILITY")

    if row["top_query_share"] >= 0.70:
        reasons.append("QUERY_CONCENTRATION")

    if row["visible_queries"] >= 10:
        reasons.append("HIGH_QUERY_COVERAGE")

    if not reasons:
        reasons.append("MONITOR")

    return ", ".join(reasons)


queue["reason_code"] = queue.apply(get_reason_codes, axis=1)


queue = queue.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

queue["priority"] = pd.cut(
    queue["model_score"],
    bins=[-np.inf, 0.50, 0.70, np.inf],
    labels=["Low", "Medium", "High"]
)

queue["recommended_action"] = queue["priority"].map({
    "High": "Review for possible content refresh",
    "Medium": "Monitor and consider review",
    "Low": "Low priority / monitor"
})

print("Queue rows:", len(queue))
print("\nTop 10 ranked actions:")

queue[
    [
        "content_hash_id",
        "model_score",
        "priority",
        "reason_code",
        "recommended_action"
    ]
].head(10)

Queue rows: 31091

Top 10 ranked actions:


,content_hash_id,model_score,priority,reason_code,recommended_action
0,content_12f8f5af4574bead,0.895,High,"DECLINE_SIGNAL, LOW_VISIBILITY, QUERY_CONCENTR...",Review for possible content refresh
1,content_ce9d662044b50053,0.895,High,"DECLINE_SIGNAL, LOW_VISIBILITY, QUERY_CONCENTR...",Review for possible content refresh
2,content_1c1b6df375d83a08,0.885,High,"DECLINE_SIGNAL, LOW_VISIBILITY, QUERY_CONCENTR...",Review for possible content refresh
3,content_376b21273b88185a,0.880,High,"DECLINE_SIGNAL, QUERY_CONCENTRATION",Review for possible content refresh
4,content_159b7b4dac8865ca,0.875,High,"DECLINE_SIGNAL, LOW_VISIBILITY, QUERY_CONCENTR...",Review for possible content refresh
5,content_99c7ddd9359238ba,0.865,High,"DECLINE_SIGNAL, QUERY_CONCENTRATION",Review for possible content refresh
6,content_b79338784b4a9f49,0.845,High,"DECLINE_SIGNAL, QUERY_CONCENTRATION",Review for possible content refresh
7,content_8180b7c0b5da3da2,0.840,High,DECLINE_SIGNAL,Review for possible content refresh
8,content_bc5db2ce76291460,0.830,High,"DECLINE_SIGNAL, QUERY_CONCENTRATION",Review for possible content refresh
9,content_7832e2403deb109e,0.810,High,"DECLINE_SIGNAL, QUERY_CONCENTRATION",Review for possible content refresh


## 2. Intended use and limits

The playbook is intended to help content and SEO teams prioritise pages for human review.

The model provides a directional signal based on the available search-performance features. It does not prove that a page will decline or explain why it declined.

The output should be used for decision-support only and should not be used to automatically change content.

In [19]:
print("Queue size:", len(queue))
print("High priority:", (queue["priority"] == "High").sum())
print("Medium priority:", (queue["priority"] == "Medium").sum())
print("Low priority:", (queue["priority"] == "Low").sum())

print("\nPriority proportions:")
print(queue["priority"].value_counts(normalize=True).round(3))

Queue size: 31091
High priority: 73
Medium priority: 609
Low priority: 30409

Priority proportions:
priority
Low       0.978
Medium    0.020
High      0.002
Name: proportion, dtype: float64


## 3. Human review + the no-go list

All recommendations should be reviewed by a human before taking action.

The reviewer should check search performance, query coverage, recent changes, seasonality and other factors not included in the model.

The model should not be used to automatically rewrite, remove or publish content, or to treat a model score as proof of a decline.

In [20]:
priority_summary = (
    queue["priority"]
    .value_counts()
    .rename_axis("priority")
    .reset_index(name="rows")
)

priority_summary["percentage"] = (
    priority_summary["rows"] / len(queue) * 100
).round(2)

print(priority_summary)

print("\nRecommended action counts:")
print(queue["recommended_action"].value_counts())

  priority   rows  percentage
0      Low  30409       97.81
1   Medium    609        1.96
2     High     73        0.23

Recommended action counts:
recommended_action
Low priority / monitor                 30409
Monitor and consider review              609
Review for possible content refresh       73
Name: count, dtype: int64


## 4. Monitoring / retrain triggers

The model should be reviewed when new data becomes available or when its performance changes noticeably.

Other triggers include major changes in feature distributions, search behaviour, or the proportion of high-priority recommendations.

Retraining should be based on a recent evaluation rather than a single unusual result.

In [21]:
current_priority = (
    queue["priority"]
    .value_counts(normalize=True)
    .reindex(["High", "Medium", "Low"], fill_value=0)
    * 100
)

print("Current priority distribution:")
for priority, proportion in current_priority.items():
    print(f"{priority}: {proportion:.2f}%")

print("\nCurrent queue size:", len(queue))
print("Current high-priority rows:", (queue["priority"] == "High").sum())

Current priority distribution:
High: 0.23%
Medium: 1.96%
Low: 97.81%

Current queue size: 31091
Current high-priority rows: 73


## 5. Exports for the paper

The ranked queue is exported so that it can be regenerated from the notebook and reused in the research paper.

The exported queue contains the content identifier, model score, priority, reason code and recommended action. It does not include client names, URLs or private search queries.

In [22]:
output_cols = [
    "content_hash_id",
    "model_score",
    "priority",
    "reason_code",
    "recommended_action"
]

queue_export = queue[output_cols].copy()

output_path = "work/outputs/content_action_queue.csv"

queue_export.to_csv(
    output_path,
    index=False
)

print("Exported:", output_path)
print("Rows:", len(queue_export))


check = pd.read_csv(output_path)

print("\nExport check:")
print("Rows:", len(check))
print("Columns:", check.columns.tolist())
print("\nTop 5 rows:")
display(check.head())

Exported: work/outputs/content_action_queue.csv
Rows: 31091

Export check:
Rows: 31091
Columns: ['content_hash_id', 'model_score', 'priority', 'reason_code', 'recommended_action']

Top 5 rows:


,content_hash_id,model_score,priority,reason_code,recommended_action
0,content_12f8f5af4574bead,0.895,High,"DECLINE_SIGNAL, LOW_VISIBILITY, QUERY_CONCENTR...",Review for possible content refresh
1,content_ce9d662044b50053,0.895,High,"DECLINE_SIGNAL, LOW_VISIBILITY, QUERY_CONCENTR...",Review for possible content refresh
2,content_1c1b6df375d83a08,0.885,High,"DECLINE_SIGNAL, LOW_VISIBILITY, QUERY_CONCENTR...",Review for possible content refresh
3,content_376b21273b88185a,0.880,High,"DECLINE_SIGNAL, QUERY_CONCENTRATION",Review for possible content refresh
4,content_159b7b4dac8865ca,0.875,High,"DECLINE_SIGNAL, LOW_VISIBILITY, QUERY_CONCENTR...",Review for possible content refresh


In [23]:
print("Files in work/outputs:")

for file in os.listdir("work/outputs"):
    print(" -", file)

print("\nQueue export exists:", os.path.exists("work/outputs/content_action_queue.csv"))

Files in work/outputs:
 - content_action_queue.csv

Queue export exists: True


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.